# Module 1: LangGraph Foundations

In this notebook, we will build your very first state machine graph from scratch. 

We will implement a linear flow that:
1. Takes a startup **topic** as input.
2. Uses an LLM to generate a **business idea** based on that topic (Node 1).
3. Uses an LLM to write a catchy **slogan** for that business idea (Node 2).
4. Connects them using edges, compiles the graph, visualizes it, and executes it.

### Step 1: Initialize Chat Model Connection

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# Load keys from .env file
load_dotenv(dotenv_path="../../langchain/.env")

model = ChatOpenAI(
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    model_name="google/gemma-2-9b-it:free",
    temperature=0.7,
)
print("Model client connected!")

---
## 1. Defining the Graph State

We define our state using Python's standard `TypedDict`. This dictionary acts as our shared database context. When any node returns updates, LangGraph updates these keys.

In [ ]:
from typing import TypedDict

class InnovationState(TypedDict):
    topic: str             # The raw theme input (e.g. "cooking")
    business_idea: str     # The generated business idea text
    slogan: str            # The generated marketing slogan

---
## 2. Creating the Nodes

Nodes are Python functions that receive the current state dictionary and return updates for specific keys. We will write two nodes that query our LLM.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

def generate_business_idea_node(state: InnovationState) -> dict:
    """Node 1: Receives the topic and generates a unique startup business idea."""
    print("--- RUNNING NODE: generate_business_idea ---")
    topic = state.get("topic")
    
    # Query LLM
    response = model.invoke([
        SystemMessage(content="You are a creative venture capitalist. Generate a unique, realistic startup business idea based on the user's topic in exactly two sentences."),
        HumanMessage(content=f"Topic: {topic}")
    ])
    
    # Return only the key we want to update
    return {"business_idea": response.content}


def generate_slogan_node(state: InnovationState) -> dict:
    """Node 2: Receives the business_idea and writes a slogan."""
    print("--- RUNNING NODE: generate_slogan ---")
    idea = state.get("business_idea")
    
    # Query LLM
    response = model.invoke([
        SystemMessage(content="You are an expert copywriter. Write a 1-sentence catchy marketing slogan for this business idea."),
        HumanMessage(content=f"Business Idea: {idea}")
    ])
    
    # Return slogan update
    return {"slogan": response.content}

---
## 3. Assembling the StateGraph

Now we register nodes and edges on the graph builder. We use `START` to indicate where the graph execution starts, and `END` to close the graph.

In [ ]:
from langgraph.graph import START, END, StateGraph

# 1. Initialize StateGraph with our state schema class
builder = StateGraph(InnovationState)

# 2. Add our node functions
builder.add_node("generate_idea", generate_business_idea_node)
builder.add_node("generate_slogan", generate_slogan_node)

# 3. Wire edges
builder.add_edge(START, "generate_idea")          # Start node points to generate_idea
builder.add_edge("generate_idea", "generate_slogan") # generate_idea points to generate_slogan
builder.add_edge("generate_slogan", END)          # generate_slogan ends the flow

# 4. Compile the graph builder into a runnable app
graph = builder.compile()
print("Graph compiled successfully!")

### Visualizing the compiled graph layout
We can display the compiled structure as an image using Mermaid.

In [ ]:
from IPython.display import Image, display

try:
    # Render graph to image binary
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # Fallback if dependencies are missing
    print(graph.get_graph().print_ascii())

---
## 4. Executing the Graph

Let's invoke the graph using `.invoke()` and print the final consolidated state dictionary.

In [ ]:
initial_state = {"topic": "remote plant watering system"}
print("Starting execution...\n")

final_result = graph.invoke(initial_state)

print("\n=== FINAL RESULTING STATE ===")
print("Topic:        ", final_result.get("topic"))
print("Business Idea:", final_result.get("business_idea"))
print("Slogan:       ", final_result.get("slogan"))

### Step-by-Step Graph Streaming (`.stream()`)

If you want to track changes as they occur at each node step, you can run `.stream()`. This yields updates containing a dictionary key representing the node name, and its returned updates: 
```
{'node_name': {'updated_key': 'value'}}
```

In [ ]:
print("--- STREAMING GRAPH STEPS ---\n")
inputs = {"topic": "gamified recycling app"}

for event in graph.stream(inputs):
    print("\nStep Event Output:")
    print(event)
    print("=" * 40)